# 第 1 章：LLM 与开发环境基础

从零开始理解 LLM 核心概念，用 LangChain 调通第一个模型。

## 1.1 LLM 是什么？

LLM（Large Language Model）是一个**文本到文本**的函数：

```python
output = llm(input)
```

类比前端：
- **Token** ≈ UTF-8 code point，但粒度更粗（1 token ≈ 0.75 英文单词）
- **Temperature** ≈ `Math.random()` 的分布宽度（0 = 确定性，1 = 最大随机）
- **Context Window** ≈ 浏览器 heap size（超出就截断）
- **System Prompt** ≈ `<meta>` 标签（全局配置）


In [ ]:
from langchain_community.chat_models.fake import FakeListChatModel

llm = FakeListChatModel(responses=["你好！我是你的 AI 助手。"])
result = llm.invoke("hello")
print(result.content)

## 1.2 System Prompt 与角色

LLM 有三种角色（类似 HTTP request/response）：

- **system**：全局指令（`<meta>` 标签）
- **user**：用户输入（HTTP request body）
- **assistant**：模型输出（HTTP response body）


In [ ]:
from langchain_community.chat_models.fake import FakeListChatModel
from langchain_core.messages import HumanMessage, SystemMessage

llm = FakeListChatModel(responses=["数学老师已就位", "7 + 5 = 12"])

# 第一轮：带 system prompt
messages = [
    SystemMessage(content="你是数学老师，只回答数学问题。"),
    HumanMessage(content="你好"),
]
r1 = llm.invoke(messages)
print(f"Round 1: {r1.content}")

# 第二轮
messages = [*messages, r1, HumanMessage(content="7+5=?")]
r2 = llm.invoke(messages)
print(f"Round 2: {r2.content}")

## 1.3 Streaming：边想边吐字

类比前端：WebSocket 推送——LLM 不是一次性返回全文，而是逐 token 流式输出。


In [ ]:
from langchain_community.chat_models.fake import FakeListChatModel

llm = FakeListChatModel(responses=["流式输出演示"])

for chunk in llm.stream("开始"):
    if chunk.content:
        print(chunk.content, end="", flush=True)
print()  # 换行

## 1.4 Token 计费

每个 LLM 调用都消耗 token，计费公式：

```python
cost = input_tokens * price_in + output_tokens * price_out
```

DeepSeek 的价格远低于 GPT-4o——这是选择默认供应商的核心原因。


In [ ]:
from cost_calculator import PRICING_TABLE, calculate_cost

# 1000 input tokens + 200 output tokens
ds_cost = calculate_cost(1000, 200, PRICING_TABLE["deepseek-chat"])
gpt_cost = calculate_cost(1000, 200, PRICING_TABLE["gpt-4o"])

print(f"DeepSeek: ${ds_cost:.6f}")
print(f"GPT-4o:   ${gpt_cost:.6f}")
print(f"DeepSeek 比 GPT-4o 便宜 {gpt_cost / ds_cost:.1f} 倍")